# YOLO12s ver6 학습 및 평가

최종 클래스: `battery`, `can`, `paper`, `pet_labeled`, `plastic`, `plastic_bag`

실행 전 Colab 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요. Roboflow에서 받은 ver6 ZIP 파일은 `/content/drive/MyDrive/cycle_pj` 또는 그 아래 `ver6` 폴더에 올려두면 자동으로 찾습니다.

In [ ]:
# 1. 필요한 패키지 설치
!pip install -q ultralytics==8.4.140 pyyaml

In [ ]:
# 2. Google Drive 연결 및 실행 환경 확인
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
import hashlib
import json
import platform
import shutil
import zipfile

import torch
import ultralytics
import yaml
from ultralytics import YOLO

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)
print('CUDA 사용 가능:', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError('GPU가 활성화되지 않았습니다. 런타임 유형을 T4 GPU로 변경하세요.')

print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 3. ver6 ZIP 자동 검색, 압축 해제, 클래스 및 경로 검사
BASE_DIR = Path('/content/drive/MyDrive/cycle_pj')
VER_DIR = BASE_DIR / 'ver6'
RESULT_DIR = BASE_DIR / 'results' / 'ver6' / 'yolo12s'
EXTRACT_DIR = Path('/content/recycle_v6_data')

VER_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

zip_files = [
    p for p in BASE_DIR.rglob('*.zip')
    if 'v6' in p.name.lower() or 'ver6' in p.name.lower()
]

if not zip_files:
    raise FileNotFoundError(
        'ver6 ZIP 파일을 찾지 못했습니다. '
        '/content/drive/MyDrive/cycle_pj 안에 업로드했는지 확인하세요.'
    )

ZIP_PATH = max(zip_files, key=lambda p: p.stat().st_mtime)
print('선택된 ZIP:', ZIP_PATH)

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

yaml_files = list(EXTRACT_DIR.rglob('data.yaml'))
if not yaml_files:
    raise FileNotFoundError('압축 파일에서 data.yaml을 찾지 못했습니다.')

ORIGINAL_DATA_YAML = min(yaml_files, key=lambda p: len(p.parts))
with open(ORIGINAL_DATA_YAML, 'r', encoding='utf-8') as f:
    data_config = yaml.safe_load(f)

raw_names = data_config.get('names')
if isinstance(raw_names, dict):
    class_names = [raw_names[k] for k in sorted(raw_names, key=lambda x: int(x))]
else:
    class_names = list(raw_names)

EXPECTED_CLASSES = {
    'battery',
    'can',
    'paper',
    'pet_labeled',
    'plastic',
    'plastic_bag',
}

print('원본 data.yaml:', ORIGINAL_DATA_YAML)
print('클래스 순서:', class_names)

if set(class_names) != EXPECTED_CLASSES:
    raise ValueError(
        '클래스 구성이 예상과 다릅니다.\n'
        f'현재 클래스: {class_names}\n'
        f'필요한 클래스: {sorted(EXPECTED_CLASSES)}'
    )

def find_split_images(folder_name):
    matches = [
        p for p in EXTRACT_DIR.rglob('images')
        if p.parent.name == folder_name
    ]
    return matches[0] if matches else None

TRAIN_IMAGES = find_split_images('train')
VAL_IMAGES = find_split_images('valid') or find_split_images('val')
TEST_IMAGES = find_split_images('test')

if TRAIN_IMAGES is None or VAL_IMAGES is None or TEST_IMAGES is None:
    raise FileNotFoundError(
        f'분할 폴더를 찾지 못했습니다. '
        f'train={TRAIN_IMAGES}, val={VAL_IMAGES}, test={TEST_IMAGES}'
    )

resolved_config = {
    'train': str(TRAIN_IMAGES),
    'val': str(VAL_IMAGES),
    'test': str(TEST_IMAGES),
    'nc': len(class_names),
    'names': class_names,
}

DATA_YAML = EXTRACT_DIR / 'data_resolved.yaml'
with open(DATA_YAML, 'w', encoding='utf-8') as f:
    yaml.safe_dump(resolved_config, f, allow_unicode=True, sort_keys=False)

print('학습용 data.yaml:', DATA_YAML)
print('Train:', TRAIN_IMAGES)
print('Validation:', VAL_IMAGES)
print('Test:', TEST_IMAGES)
print('데이터셋 및 클래스 검사 완료')

In [ ]:
# 4. Train/Valid/Test 이미지와 라벨 감사
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
split_image_dirs = {
    'train': TRAIN_IMAGES,
    'valid': VAL_IMAGES,
    'test': TEST_IMAGES,
}

zip_hasher = hashlib.sha256()
with open(ZIP_PATH, 'rb') as zip_file:
    for chunk in iter(lambda: zip_file.read(1024 * 1024), b''):
        zip_hasher.update(chunk)

audit = {
    'zip_sha256': zip_hasher.hexdigest(),
    'zip_path': str(ZIP_PATH),
    'data_yaml': str(DATA_YAML),
    'class_names': class_names,
    'splits': {},
}

hash_locations = defaultdict(list)
converted_polygons = 0
original_boxes = 0

for split_name, image_dir in split_image_dirs.items():
    label_dir = image_dir.parent / 'labels'
    images = sorted([
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ])

    class_counts = Counter()
    missing_label_files = []
    empty_label_files = []
    invalid_labels = []

    for image_path in images:
        file_hash = hashlib.sha256(image_path.read_bytes()).hexdigest()
        hash_locations[file_hash].append({
            'split': split_name,
            'file': image_path.name,
        })

        label_path = label_dir / f'{image_path.stem}.txt'
        if not label_path.exists():
            missing_label_files.append(image_path.name)
            continue

        lines = [line.strip() for line in label_path.read_text(encoding='utf-8').splitlines() if line.strip()]
        if not lines:
            empty_label_files.append(label_path.name)
            continue

        converted_lines = []
        for line_number, line in enumerate(lines, start=1):
            parts = line.split()
            if len(parts) < 5:
                invalid_labels.append({
                    'file': label_path.name,
                    'line': line_number,
                    'reason': 'YOLO 라벨 항목 수가 너무 적음',
                })
                continue

            try:
                class_id = int(float(parts[0]))
                coordinates = [float(value) for value in parts[1:]]
            except ValueError:
                invalid_labels.append({
                    'file': label_path.name,
                    'line': line_number,
                    'reason': '숫자 변환 실패',
                })
                continue

            if not 0 <= class_id < len(class_names):
                invalid_labels.append({
                    'file': label_path.name,
                    'line': line_number,
                    'reason': f'잘못된 class_id={class_id}',
                })
                continue

            if any(value < 0 or value > 1 for value in coordinates):
                invalid_labels.append({
                    'file': label_path.name,
                    'line': line_number,
                    'reason': '좌표가 0~1 범위를 벗어남',
                })
                continue

            if len(coordinates) == 4:
                x_center, y_center, width, height = coordinates
                original_boxes += 1
            elif len(coordinates) >= 6 and len(coordinates) % 2 == 0:
                xs = coordinates[0::2]
                ys = coordinates[1::2]
                x_min, x_max = min(xs), max(xs)
                y_min, y_max = min(ys), max(ys)
                x_center = (x_min + x_max) / 2
                y_center = (y_min + y_max) / 2
                width = x_max - x_min
                height = y_max - y_min
                converted_polygons += 1
            else:
                invalid_labels.append({
                    'file': label_path.name,
                    'line': line_number,
                    'reason': f'지원하지 않는 좌표 개수={len(coordinates)}',
                })
                continue

            if width <= 0 or height <= 0:
                invalid_labels.append({
                    'file': label_path.name,
                    'line': line_number,
                    'reason': '바운딩박스 너비 또는 높이가 0 이하',
                })
                continue

            converted_lines.append(
                f'{class_id} {x_center:.8f} {y_center:.8f} {width:.8f} {height:.8f}'
            )
            class_counts[class_names[class_id]] += 1

        if converted_lines:
            label_path.write_text('\n'.join(converted_lines) + '\n', encoding='utf-8')

    audit['splits'][split_name] = {
        'images': len(images),
        'label_files': len(list(label_dir.glob('*.txt'))) if label_dir.exists() else 0,
        'objects_per_class': dict(class_counts),
        'missing_label_count': len(missing_label_files),
        'empty_label_count': len(empty_label_files),
        'invalid_label_count': len(invalid_labels),
        'missing_label_examples': missing_label_files[:20],
        'empty_label_examples': empty_label_files[:20],
        'invalid_label_examples': invalid_labels[:20],
    }

cross_split_duplicates = []
for locations in hash_locations.values():
    involved_splits = {item['split'] for item in locations}
    if len(involved_splits) > 1:
        cross_split_duplicates.append(locations)

audit['converted_polygons'] = converted_polygons
audit['original_boxes'] = original_boxes
audit['cross_split_duplicate_groups'] = len(cross_split_duplicates)
audit['cross_split_duplicate_examples'] = cross_split_duplicates[:20]

AUDIT_PATH = RESULT_DIR / 'dataset_audit.json'
with open(AUDIT_PATH, 'w', encoding='utf-8') as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)

print(json.dumps(audit, ensure_ascii=False, indent=2))
print('감사 결과 저장:', AUDIT_PATH)

total_invalid = sum(
    info.get('missing_label_count', 0) + info.get('invalid_label_count', 0)
    for info in audit['splits'].values()
)
if total_invalid > 0:
    raise ValueError('라벨 문제가 발견됐습니다. 위 출력과 dataset_audit.json을 확인하세요.')

print('데이터셋 감사 완료: 치명적인 라벨 문제 없음')

## 학습 전 확인

위 출력에서 여섯 클래스 이름과 각 분할의 수량이 정상인지 확인한 후 아래 학습 셀을 실행하세요. 기존 ver5의 `best.pt`를 이어서 학습하지 않고 공식 `yolo12s.pt`에서 시작합니다.

In [ ]:
# 5. YOLO12s 학습
RUN_NAME = datetime.now().strftime('train_%Y%m%d_%H%M%S')

model = YOLO('yolo12s.pt')

train_result = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=8,
    seed=42,
    deterministic=True,
    device=0,
    workers=2,
    patience=0,
    cache=False,
    amp=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.9,
    weight_decay=0.0005,
    plots=True,
    project=str(RESULT_DIR),
    name=RUN_NAME,
    exist_ok=True,
)

RUN_DIR = Path(train_result.save_dir)
BEST_PATH = RUN_DIR / 'weights' / 'best.pt'

if not BEST_PATH.exists():
    raise FileNotFoundError(f'학습은 끝났지만 best.pt를 찾지 못했습니다: {BEST_PATH}')

print('학습 결과 폴더:', RUN_DIR)
print('best.pt:', BEST_PATH)

In [ ]:
# 6. Test split 정량 평가 및 conf=0.25 혼동행렬 생성
best_model = YOLO(str(BEST_PATH))

# mAP, Precision, Recall 계산용 평가
metrics = best_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=640,
    batch=8,
    conf=0.001,
    iou=0.7,
    device=0,
    plots=True,
    project=str(RUN_DIR),
    name='test_eval',
    exist_ok=True,
)

TEST_EVAL_DIR = RUN_DIR / 'test_eval'

test_metrics = {
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
    'mAP50': float(metrics.box.map50),
    'mAP50_95': float(metrics.box.map),
    'per_class_mAP50_95': {
        str(best_model.names[i]): float(value)
        for i, value in enumerate(metrics.box.maps)
    },
    'speed_ms': {
        key: float(value) for key, value in metrics.speed.items()
    },
}

with open(TEST_EVAL_DIR / 'test_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)

# 실제 운용 임계값 conf=0.25 기준 혼동행렬
best_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=640,
    batch=8,
    conf=0.25,
    iou=0.7,
    device=0,
    plots=True,
    project=str(RUN_DIR),
    name='test_eval_conf025',
    exist_ok=True,
)

FINAL_BEST_PATH = VER_DIR / 'yolo12s_best.pt'
shutil.copy2(BEST_PATH, FINAL_BEST_PATH)

environment = {
    'python': platform.python_version(),
    'ultralytics': ultralytics.__version__,
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
}

with open(RUN_DIR / 'environment.json', 'w', encoding='utf-8') as f:
    json.dump(environment, f, ensure_ascii=False, indent=2)

run_summary = {
    'run_dir': str(RUN_DIR),
    'best_path': str(BEST_PATH),
    'final_best_path': str(FINAL_BEST_PATH),
    'dataset_audit': str(AUDIT_PATH),
    'test_metrics': str(TEST_EVAL_DIR / 'test_metrics.json'),
    'conf025_dir': str(RUN_DIR / 'test_eval_conf025'),
}

with open(RUN_DIR / 'run_summary.json', 'w', encoding='utf-8') as f:
    json.dump(run_summary, f, ensure_ascii=False, indent=2)

print(json.dumps(test_metrics, ensure_ascii=False, indent=2))
print('최종 모델:', FINAL_BEST_PATH)
print('전체 학습 결과:', RUN_DIR)
print('기본 Test 평가:', TEST_EVAL_DIR)
print('conf=0.25 평가:', RUN_DIR / 'test_eval_conf025')

In [ ]:
# 7. 선택 사항: Google Drive의 새 이미지 폴더 예측
# 사용할 경우 cycle_pj/ver6/new_images 폴더를 만들고 이미지를 넣으세요.
NEW_IMAGES_DIR = VER_DIR / 'new_images'

if NEW_IMAGES_DIR.exists() and any(NEW_IMAGES_DIR.iterdir()):
    prediction_results = best_model.predict(
        source=str(NEW_IMAGES_DIR),
        imgsz=640,
        conf=0.25,
        iou=0.7,
        device=0,
        save=True,
        save_txt=True,
        save_conf=True,
        project=str(RUN_DIR),
        name='new_image_test',
        exist_ok=True,
    )
    print('새 이미지 결과:', RUN_DIR / 'new_image_test')
else:
    print('새 이미지 폴더가 비어 있어 건너뜁니다:', NEW_IMAGES_DIR)

## 최종 저장 위치

- 최종 모델: `/content/drive/MyDrive/cycle_pj/ver6/yolo12s_best.pt`
- 학습 결과: `/content/drive/MyDrive/cycle_pj/results/ver6/yolo12s/train_날짜_시간/`
- Test 지표: `test_eval/test_metrics.json`
- conf=0.25 혼동행렬: `test_eval_conf025/confusion_matrix.png`
- 데이터 감사: `/content/drive/MyDrive/cycle_pj/results/ver6/yolo12s/dataset_audit.json`